# 01 — Extract (Task 1) — نسخة مبنية على API رسمي مكتشف

**اكتشاف مهم:** منصة اعتماد عندها endpoint داخلي يرجّع JSON جاهز، لقيناه عبر Network tab بالمتصفح:
```
https://tenders.etimad.sa/Tender/AllSupplierTendersForVisitorAsync?PageSize=6&PublishDateId=5&pageNumber=1
```
هذا معناه **ما نحتاج Playwright ولا متصفح آلي ولا selectors إطلاقًا** — بس `requests` عادي.
كذا حلينا تلقائيًا: مشكلة Akamai (الـ API نفسه ما يحظرنا)، مشكلة التكرار (نستخدم `tenderId` الفريد
من الـ API مباشرة بدل التخمين)، ومشكلة الهمزة بالتواريخ (التواريخ بصيغة ISO نظيفة، مالها علاقة بالنص العربي).

هذا الـ notebook يحفظ كل سجل **خام كما رجع من الـ API** بدون أي تعديل — التنظيف يصير بـ Task 2.


### 1. المكتبات

In [ ]:
import requests
import json
import time
from datetime import date
from pathlib import Path


### 2. إعدادات ثابتة

- `BASE_URL`: رابط الـ API المكتشف
- `PAGE_SIZE`: عدد المناقصات بكل صفحة/طلب — بما إنه API مباشر (مو متصفح)، نقدر نطلب دفعات أكبر بكثير من 6
- `PAGES_TO_SCRAPE`: عدد الطلبات (كل طلب = صفحة) — ابدئي بعدد صغير للتجربة


In [ ]:
BASE_URL = "https://tenders.etimad.sa/Tender/AllSupplierTendersForVisitorAsync"
PAGE_SIZE = 100  # جرّبي هذا الرقم؛ لو رجع خطأ من السيرفر قلّليه (مثلاً لـ 50 أو 24)
PAGES_TO_SCRAPE = 20  # TODO: زيديها تدريجيًا بعد ما تتأكدين كل شي سليم
PUBLISH_DATE_ID = 5  # TODO: هذا الرقم طلع بالرابط الأصلي اللي لقيناه — لو غيّرتيه شوفي هل يأثر على الفلترة

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
today_str = date.today().isoformat()

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://tenders.etimad.sa/Tender/AllTendersForVisitor?PageNumber=1",
}


### 3. طلب صفحة وحدة من الـ API

In [ ]:
def fetch_page(page_number, page_size=PAGE_SIZE):
    params = {
        "PageSize": page_size,
        "PublishDateId": PUBLISH_DATE_ID,
        "pageNumber": page_number,
    }
    response = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return response.json()


### 4. اكتشاف شكل البيانات (خلية تشخيصية — شغّليها أول شي)

بما إننا ما شفنا شكل الـ JSON كامل من الأول، هذي الخلية تطلع لك:
- نوع البيانات اللي رجعت (قائمة مباشرة، أو Dictionary فيه القائمة جوا مفتاح معيّن)
- كل أسماء الحقول (keys) الموجودة بأول سجل — عشان تشوفين وين حقل اسم الجهة بالضبط (مهم لـ Task 2)

**شغّلي هذي الخلية وطلعي لي شو طبعت، خصوصًا قائمة الـ keys.**


In [ ]:
sample = fetch_page(page_number=1, page_size=5)

print("نوع البيانات:", type(sample))

items_preview = []

if isinstance(sample, list):
    items_preview = sample
    print(f"عدد العناصر بأول صفحة: {len(items_preview)}")
elif isinstance(sample, dict):
    print("مفاتيح المستوى الأول:", list(sample.keys()))
    for key in ["data", "Data", "result", "Result", "items", "Items", "tenders"]:
        if key in sample and isinstance(sample[key], list):
            items_preview = sample[key]
            print(f"لقينا القائمة تحت المفتاح: '{key}' — عدد العناصر: {len(items_preview)}")
            break
    else:
        print("ما قدرنا نحدد تلقائيًا وين القائمة — شوفي المفاتيح فوق يدويًا")

if items_preview:
    print("\nأسماء الحقول (keys) بأول سجل:")
    print(list(items_preview[0].keys()))
    print("\nعيّنة من أول سجل:")
    print(json.dumps(items_preview[0], ensure_ascii=False, indent=2))


### 5. دالة موحّدة لاستخراج القائمة من أي شكل استجابة

بناءً على خلية التشخيص فوق، عدّلي `RESPONSE_LIST_KEY` إذا كانت القائمة جوا مفتاح معيّن،
أو سيبيها `None` إذا كانت الاستجابة قائمة مباشرة.


In [ ]:
RESPONSE_LIST_KEY = "data"  # ✅ مؤكد من التشخيص


def extract_items(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and RESPONSE_LIST_KEY:
        return payload.get(RESPONSE_LIST_KEY, [])
    return []


def get_total_count(payload):
    """الاستجابة فيها 'totalCount' — نستخدمه لنعرف بالضبط كم صفحة موجودة، بدل التخمين."""
    if isinstance(payload, dict):
        return payload.get("totalCount")
    return None


### 6. سحب كل الصفحات وتجميعها

نستخدم `tenderId` كمعرّف فريد لمنع التكرار — مباشرة من الـ API، بدون أي تخمين أو Regex.
كمان نطبع `totalCount` بأول صفحة عشان تعرفين كم إجمالي المناقصات المتاحة كلها (مو بس اللي سحبناها).


In [ ]:
def run_extraction(pages_to_scrape):
    all_items = []
    seen_ids = set()
    total_available = None

    for page_number in range(1, pages_to_scrape + 1):
        payload = fetch_page(page_number)
        items = extract_items(payload)

        if page_number == 1:
            total_available = get_total_count(payload)
            if total_available is not None:
                print(f"إجمالي المناقصات المتاحة بالمنصة كلها: {total_available}")
                print(f"(يعني لو PAGE_SIZE={PAGE_SIZE}، نحتاج تقريبًا {-(-total_available // PAGE_SIZE)} صفحة لسحبها كلها)\n")

        if not items:
            print(f"صفحة {page_number}: ما رجع فيها بيانات — نوقف هنا (يمكن وصلنا آخر صفحة)")
            break

        new_count = 0
        for item in items:
            tender_id = item.get("tenderId")
            if tender_id is not None and tender_id not in seen_ids:
                seen_ids.add(tender_id)
                all_items.append(item)
                new_count += 1

        print(f"صفحة {page_number}: {len(items)} عنصر، منهم {new_count} جديد (غير مكرر)")
        time.sleep(1)  # مجاملة للسيرفر، تجنّب الضغط عليه بسرعة كبيرة

    return all_items


records = run_extraction(PAGES_TO_SCRAPE)
print(f"\nإجمالي السجلات الفريدة المستخرجة: {len(records)}")


### 7. حفظ خام في `data/raw/`

نحفظ كل سجل **كما رجع من الـ API تمامًا** بدون أي تعديل أو حذف حقول — القرار بأي حقول نستخدم
ونصنّف يصير بـ Task 2، مو هنا.


In [ ]:
output_path = RAW_DIR / f"etimad_all_tenders_{today_str}.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"تم الحفظ في: {output_path}")


### 8. نظرة سريعة على التنوّع (لأغراض العرض لاحقًا — التصنيف الدقيق بـ Task 2)

`AGENCY_FIELD_NAME` مؤكد من التشخيص = `agencyName` ✅


In [ ]:
AGENCY_FIELD_NAME = "agencyName"  # ✅ مؤكد من التشخيص

distinct_agencies = sorted(set(
    r.get(AGENCY_FIELD_NAME) for r in records if r.get(AGENCY_FIELD_NAME)
))
distinct_activities = sorted(set(
    r.get("tenderActivityName") for r in records if r.get("tenderActivityName")
))

print(f"إجمالي المناقصات: {len(records)}")
print(f"عدد الجهات الفريدة (خام): {len(distinct_agencies)}")
print(f"عدد الأنشطة/الفئات الفريدة: {len(distinct_activities)}")


---
## قبل ما تشغّلين هذا فعليًا

1. `pip install requests` (لو ماهي مثبتة أصلًا) ✅ عندك مثبتة
2. شغّلي الخلايا بالترتيب من فوق لتحت
3. خطوة 5 و8 مثبّتة الحين (`RESPONSE_LIST_KEY = "data"`, `AGENCY_FIELD_NAME = "agencyName"`) — مؤكدة من التشخيص، ما تحتاجين تعدّلين شي
4. راقبي `totalCount` اللي يطبع بخطوة 6 — يعطيك فكرة عن حجم البيانات الكلي بالمنصة
5. لو رجع خطأ 400/403 من السيرفر، جرّبي تقلّلين `PAGE_SIZE`، أو تأكدي إن `HEADERS` معبّية صح
